# 2. Data Cleaning and Audit

**Goal:** convert the raw dataset into one clean, chronologically ordered dataset for later EDA and modeling.

**Decision:** remove missing/infinite values and exact duplicate rows. Do not scale, transform, encode, or remove time. Do not remove legitimate or fraud examples beyond the explicit row-quality rules.

## 2.1 Project setup

**Purpose:** import shared paths and required libraries.

**Process:** locate the project from local or Colab storage and load `src/config.py`.

**Output:** confirmed raw, processed, and report paths.

In [1]:
import os
import sys
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/card-fraud-triage')
    IN_COLAB = True
except ImportError:
    PROJECT_ROOT = Path.cwd().resolve()
    if not (PROJECT_ROOT / 'src/config.py').is_file():
        PROJECT_ROOT = PROJECT_ROOT.parent
    IN_COLAB = False

if not (PROJECT_ROOT / 'src/config.py').is_file():
    raise FileNotFoundError('Could not locate src/config.py.')
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

import json
import random
import numpy as np
import pandas as pd

from src.config import ARTIFACTS, CLEAN_PATH, RAW_PATH, SEED

random.seed(SEED)
pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', lambda value: '{:,.4f}'.format(value))

RAW_DATA_PATH = Path(RAW_PATH)
CLEAN_DATA_PATH = Path(CLEAN_PATH)
REPORT_PATH = Path(ARTIFACTS) / 'clean_report.json'

print('Environment:', 'Google Colab' if IN_COLAB else 'Local')
print('Raw:', RAW_DATA_PATH)
print('Clean:', CLEAN_DATA_PATH)

Environment: Local
Raw: /Users/yuvrajchandravansi/Desktop/card-fraud-triage/data/raw/creditcard.csv
Clean: /Users/yuvrajchandravansi/Desktop/card-fraud-triage/data/processed/creditcard_clean.csv


## 2.2 Load raw data

**Purpose:** load the source independently of the extraction notebook.

**Process:** read `RAW_PATH` and preserve the original dataframe.

**Output:** raw row count, column count, and a sample.

In [2]:
if not RAW_DATA_PATH.is_file():
    raise FileNotFoundError('Raw dataset missing: {}'.format(RAW_DATA_PATH))

df_raw = pd.read_csv(RAW_DATA_PATH)
print('Raw shape: {:,} rows x {} columns'.format(*df_raw.shape))
display(df_raw.head())

Raw shape: 284,807 rows x 31 columns


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0000,-1.3598,-0.0728,2.5363,1.3782,-0.3383,0.4624,0.2396,0.0987,0.3638,0.0908,-0.5516,-0.6178,-0.9914,-0.3112,1.4682,-0.4704,0.2080,0.0258,0.4040,0.2514,-0.0183,0.2778,-0.1105,0.0669,0.1285,-0.1891,0.1336,-0.0211,149.6200,0
1,0.0000,1.1919,0.2662,0.1665,0.4482,0.0600,-0.0824,-0.0788,0.0851,-0.2554,-0.1670,1.6127,1.0652,0.4891,-0.1438,0.6356,0.4639,-0.1148,-0.1834,-0.1458,-0.0691,-0.2258,-0.6387,0.1013,-0.3398,0.1672,0.1259,-0.0090,0.0147,2.6900,0
2,1.0000,-1.3584,-1.3402,1.7732,0.3798,-0.5032,1.8005,0.7915,0.2477,-1.5147,0.2076,0.6245,0.0661,0.7173,-0.1659,2.3459,-2.8901,1.1100,-0.1214,-2.2619,0.5250,0.2480,0.7717,0.9094,-0.6893,-0.3276,-0.1391,-0.0554,-0.0598,378.6600,0
3,1.0000,-0.9663,-0.1852,1.7930,-0.8633,-0.0103,1.2472,0.2376,0.3774,-1.3870,-0.0550,-0.2265,0.1782,0.5078,-0.2879,-0.6314,-1.0596,-0.6841,1.9658,-1.2326,-0.2080,-0.1083,0.0053,-0.1903,-1.1756,0.6474,-0.2219,0.0627,0.0615,123.5000,0
4,2.0000,-1.1582,0.8777,1.5487,0.4030,-0.4072,0.0959,0.5929,-0.2705,0.8177,0.7531,-0.8228,0.5382,1.3459,-1.1197,0.1751,-0.4514,-0.2370,-0.0382,0.8035,0.4085,-0.0094,0.7983,-0.1375,0.1413,-0.2060,0.5023,0.2194,0.2152,69.9900,0


## 2.3 Validate schema and labels

**Purpose:** fail early if required data or target labels are invalid.

**Process:** check required columns, unexpected duplicate names, numeric types, `Class` values, negative `Amount`, and infinite values.

**Output:** one pass/fail validation table. Invalid input raises a clear error.

In [3]:
expected_columns = ['Time'] + ['V{}'.format(i) for i in range(1, 29)] + ['Amount', 'Class']
missing_columns = [column for column in expected_columns if column not in df_raw.columns]
extra_columns = [column for column in df_raw.columns if column not in expected_columns]
duplicate_names = int(df_raw.columns.duplicated().sum())
non_numeric = [column for column in df_raw.columns if not pd.api.types.is_numeric_dtype(df_raw[column])]
invalid_classes = sorted(set(df_raw['Class'].dropna().unique()) - {0, 1}) if 'Class' in df_raw else ['missing Class']
negative_amount = int((df_raw['Amount'].dropna() < 0).sum()) if 'Amount' in df_raw else len(df_raw)
numeric = df_raw.select_dtypes(include=np.number)
infinite_count = int(np.isinf(numeric.to_numpy()).sum())

checks = pd.DataFrame([
    ('Required columns missing', missing_columns),
    ('Unexpected columns', extra_columns),
    ('Duplicate column names', duplicate_names),
    ('Non-numeric columns', non_numeric),
    ('Invalid Class values', invalid_classes),
    ('Negative Amount rows', negative_amount),
    ('Infinite numeric values', infinite_count),
], columns=['check', 'observed'])
checks['passed'] = [
    not missing_columns, not extra_columns, duplicate_names == 0, not non_numeric,
    not invalid_classes, negative_amount == 0, infinite_count == 0,
]
display(checks)

failures = checks.loc[~checks['passed'], 'check'].tolist()
if failures:
    raise ValueError('Validation failed: {}'.format(failures))
print('Schema and target validation passed.')

,check,observed,passed
0,Required columns missing,[],True
1,Unexpected columns,[],True
2,Duplicate column names,0,True
3,Non-numeric columns,[],True
4,Invalid Class values,[],True
5,Negative Amount rows,0,True
6,Infinite numeric values,0,True


Schema and target validation passed.


## 2.4 Record the pre-cleaning baseline

**Purpose:** create a reproducible before-state for the audit report.

**Process:** count rows, missing cells, infinite values, exact duplicates, frauds, and zero amounts.

**Output:** a before-cleaning summary table.

In [4]:
before = pd.Series({
    'Rows': len(df_raw),
    'Columns': len(df_raw.columns),
    'Missing cells': int(df_raw.isna().sum().sum()),
    'Infinite numeric cells': infinite_count,
    'Exact duplicate rows': int(df_raw.duplicated().sum()),
    'Legitimate transactions': int((df_raw['Class'] == 0).sum()),
    'Fraud transactions': int((df_raw['Class'] == 1).sum()),
    'Fraud rate (%)': float(df_raw['Class'].mean() * 100),
}, name='Before cleaning')
display(before.to_frame())

,Before cleaning
Rows,"284,807.0000"
Columns,31.0000
Missing cells,0.0000
Infinite numeric cells,0.0000
Exact duplicate rows,"1,081.0000"
Legitimate transactions,"284,315.0000"
Fraud transactions,492.0000
Fraud rate (%),0.1727


## 2.5 Review duplicate rows

**Purpose:** understand what is removed and confirm that duplicate labels are not conflicting.

**Process:** identify exact duplicate records and any identical feature groups with different labels.

**Output:** duplicate counts by class and the number of conflicting label groups.

In [5]:
duplicate_mask = df_raw.duplicated(keep='first')
duplicate_rows = df_raw.loc[duplicate_mask]
duplicate_frauds = int((duplicate_rows['Class'] == 1).sum())
duplicate_legitimate = int((duplicate_rows['Class'] == 0).sum())

feature_columns = [column for column in df_raw.columns if column != 'Class']
groups = df_raw.loc[df_raw.duplicated(subset=feature_columns, keep=False)].groupby(feature_columns, dropna=False)['Class'].nunique(dropna=False)
conflicting_groups = int((groups > 1).sum())

duplicate_audit = pd.DataFrame({
    'metric': ['Exact duplicate rows', 'Duplicate fraud rows', 'Duplicate legitimate rows', 'Identical-feature groups with conflicting labels'],
    'count': [len(duplicate_rows), duplicate_frauds, duplicate_legitimate, conflicting_groups],
}).set_index('metric')
display(duplicate_audit)
print('Policy: remove only extra copies of exact duplicate rows; never remove conflicting evidence automatically.')

,count
metric,
Exact duplicate rows,1081
Duplicate fraud rows,19
Duplicate legitimate rows,1062
Identical-feature groups with conflicting labels,0


Policy: remove only extra copies of exact duplicate rows; never remove conflicting evidence automatically.


## 2.6 Remove missing, infinite, and duplicate rows

**Purpose:** apply the two explicit quality rules.

**Process:** convert infinite numeric values to missing, drop rows containing missing values, then remove exact duplicate copies.

**Output:** separate removal counts for each cleaning rule.

In [6]:
df_clean = df_raw.copy()
numeric_columns = df_clean.select_dtypes(include=np.number).columns
df_clean[numeric_columns] = df_clean[numeric_columns].replace([np.inf, -np.inf], np.nan)

rows_before_missing = len(df_clean)
df_clean = df_clean.dropna().copy()
missing_or_infinite_removed = rows_before_missing - len(df_clean)

rows_before_duplicates = len(df_clean)
df_clean = df_clean.drop_duplicates(keep='first').copy()
duplicate_removed = rows_before_duplicates - len(df_clean)

print('Rows removed for missing/infinite values:', missing_or_infinite_removed)
print('Exact duplicate rows removed:', duplicate_removed)
print('Rows retained:', len(df_clean))

Rows removed for missing/infinite values: 0
Exact duplicate rows removed: 1081
Rows retained: 283726


## 2.7 Sort chronologically and restore the original schema

**Purpose:** make downstream chronological splits reliable.

**Process:** keep original column order, sort by `Time` ascending with a stable sort, reset the index, and retain `Time`.

**Output:** a chronological dataset. No feature engineering or scaling occurs.

In [7]:
df_clean = df_clean.loc[:, df_raw.columns].sort_values('Time', kind='stable').reset_index(drop=True)
df_clean['Class'] = df_clean['Class'].astype('int8')
print('Chronological range: {:.0f} to {:.0f} seconds'.format(df_clean['Time'].min(), df_clean['Time'].max()))
display(df_clean.head())

Chronological range: 0 to 172792 seconds


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0000,-1.3598,-0.0728,2.5363,1.3782,-0.3383,0.4624,0.2396,0.0987,0.3638,0.0908,-0.5516,-0.6178,-0.9914,-0.3112,1.4682,-0.4704,0.2080,0.0258,0.4040,0.2514,-0.0183,0.2778,-0.1105,0.0669,0.1285,-0.1891,0.1336,-0.0211,149.6200,0
1,0.0000,1.1919,0.2662,0.1665,0.4482,0.0600,-0.0824,-0.0788,0.0851,-0.2554,-0.1670,1.6127,1.0652,0.4891,-0.1438,0.6356,0.4639,-0.1148,-0.1834,-0.1458,-0.0691,-0.2258,-0.6387,0.1013,-0.3398,0.1672,0.1259,-0.0090,0.0147,2.6900,0
2,1.0000,-1.3584,-1.3402,1.7732,0.3798,-0.5032,1.8005,0.7915,0.2477,-1.5147,0.2076,0.6245,0.0661,0.7173,-0.1659,2.3459,-2.8901,1.1100,-0.1214,-2.2619,0.5250,0.2480,0.7717,0.9094,-0.6893,-0.3276,-0.1391,-0.0554,-0.0598,378.6600,0
3,1.0000,-0.9663,-0.1852,1.7930,-0.8633,-0.0103,1.2472,0.2376,0.3774,-1.3870,-0.0550,-0.2265,0.1782,0.5078,-0.2879,-0.6314,-1.0596,-0.6841,1.9658,-1.2326,-0.2080,-0.1083,0.0053,-0.1903,-1.1756,0.6474,-0.2219,0.0627,0.0615,123.5000,0
4,2.0000,-1.1582,0.8777,1.5487,0.4030,-0.4072,0.0959,0.5929,-0.2705,0.8177,0.7531,-0.8228,0.5382,1.3459,-1.1197,0.1751,-0.4514,-0.2370,-0.0382,0.8035,0.4085,-0.0094,0.7983,-0.1375,0.1413,-0.2060,0.5023,0.2194,0.2152,69.9900,0


## 2.8 Verify the cleaned dataset

**Purpose:** stop if the cleaned file violates a required invariant.

**Process:** assert no missing values, no infinite values, no exact duplicates, ascending time, valid classes, and non-negative amounts.

**Output:** explicit pass/fail checks before saving.

In [8]:
numeric_clean = df_clean.select_dtypes(include=np.number).to_numpy()
final_checks = pd.DataFrame([
    ('No missing values', int(df_clean.isna().sum().sum()) == 0),
    ('No infinite values', not np.isinf(numeric_clean).any()),
    ('No exact duplicate rows', not df_clean.duplicated().any()),
    ('Time sorted ascending', df_clean['Time'].is_monotonic_increasing),
    ('Class in {0, 1}', set(df_clean['Class'].unique()).issubset({0, 1})),
    ('Amount non-negative', (df_clean['Amount'] >= 0).all()),
], columns=['check', 'passed'])
display(final_checks)
if not final_checks['passed'].all():
    raise AssertionError('Cleaned-data invariant failed.')
print('All cleaning checks passed.')

,check,passed
0,No missing values,True
1,No infinite values,True
2,No exact duplicate rows,True
3,Time sorted ascending,True
4,"Class in {0, 1}",True
5,Amount non-negative,True


All cleaning checks passed.


## 2.9 Compare before and after

**Purpose:** show the effect of cleaning in one compact table.

**Process:** compare rows, columns, duplicates, nulls, fraud count, and fraud rate.

**Output:** a before/after audit table.

In [9]:
after = pd.Series({
    'Rows': len(df_clean),
    'Columns': len(df_clean.columns),
    'Missing cells': int(df_clean.isna().sum().sum()),
    'Exact duplicate rows': int(df_clean.duplicated().sum()),
    'Legitimate transactions': int((df_clean['Class'] == 0).sum()),
    'Fraud transactions': int((df_clean['Class'] == 1).sum()),
    'Fraud rate (%)': float(df_clean['Class'].mean() * 100),
}, name='After cleaning')
comparison = pd.concat([before, after], axis=1)
display(comparison)
print('Total rows removed:', len(df_raw) - len(df_clean))

,Before cleaning,After cleaning
Rows,"284,807.0000","283,726.0000"
Columns,31.0000,31.0000
Missing cells,0.0000,0.0000
Infinite numeric cells,0.0000,NaN
Exact duplicate rows,"1,081.0000",0.0000
Legitimate transactions,"284,315.0000","283,253.0000"
Fraud transactions,492.0000,473.0000
Fraud rate (%),0.1727,0.1667


Total rows removed: 1081


## 2.10 Save cleaned data and audit report

**Purpose:** persist the cleaned dataset and evidence for the EDA notebook.

**Process:** write CSV to `CLEAN_PATH`, then write a JSON report containing removal counts and before/after totals.

**Output:** `creditcard_clean.csv` and `clean_report.json`.

In [10]:
CLEAN_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(CLEAN_DATA_PATH, index=False)

clean_report = {
    'project': 'Card Fraud Triage',
    'raw_dataset': str(RAW_DATA_PATH),
    'clean_dataset': str(CLEAN_DATA_PATH),
    'raw_shape': [int(df_raw.shape[0]), int(df_raw.shape[1])],
    'clean_shape': [int(df_clean.shape[0]), int(df_clean.shape[1])],
    'rows_removed': {
        'missing_or_infinite_values': int(missing_or_infinite_removed),
        'exact_duplicates': int(duplicate_removed),
        'total': int(len(df_raw) - len(df_clean)),
    },
    'removed': {
        'missing_values': int(missing_or_infinite_removed),
        'rows_with_missing_values': int(missing_or_infinite_removed),
        'duplicate_rows': int(duplicate_removed),
    },
    'duplicate_analysis': {
        'duplicate_rows_labelled_fraud': duplicate_frauds,
        'duplicate_rows_labelled_legitimate': duplicate_legitimate,
        'identical_feature_groups_with_conflicting_labels': conflicting_groups,
    },
    'before_cleaning': {
        'Rows': int(before['Rows']), 'Columns': int(before['Columns']),
        'Legitimate transactions': int(before['Legitimate transactions']),
        'Fraud transactions': int(before['Fraud transactions']),
        'Fraud rate (%)': float(before['Fraud rate (%)']),
        'Missing cells': int(before['Missing cells']),
        'Exact duplicate rows': int(before['Exact duplicate rows']),
    },
    'after_cleaning': {
        'Rows': int(after['Rows']), 'Columns': int(after['Columns']),
        'Legitimate transactions': int(after['Legitimate transactions']),
        'Fraud transactions': int(after['Fraud transactions']),
        'Fraud rate (%)': float(after['Fraud rate (%)']),
        'Missing cells': int(after['Missing cells']),
        'Exact duplicate rows': int(after['Exact duplicate rows']),
    },
    'before': {
        'rows': int(len(df_raw)), 'duplicates': int(before['Exact duplicate rows']),
        'nulls': int(before['Missing cells']), 'frauds': int(before['Fraud transactions']),
        'fraud_rate': float(df_raw['Class'].mean()),
    },
    'after': {
        'rows': int(len(df_clean)), 'duplicates': 0,
        'nulls': 0, 'frauds': int(after['Fraud transactions']),
        'fraud_rate': float(df_clean['Class'].mean()),
    },
    'zero_amount_rows_retained': int((df_clean['Amount'] == 0).sum()),
    'outlier_policy': 'IQR flags are diagnostic only; no outliers were removed.',
    'preprocessing_policy': 'Scaling and train/validation/test splitting are intentionally deferred.',
    'modeling_note': 'Exclude Class from model features. Use Time only for chronological ordering.',
}
REPORT_PATH.write_text(json.dumps(clean_report, indent=2) + chr(10), encoding='utf-8')

print('Saved cleaned dataset:', CLEAN_DATA_PATH)
print('Saved cleaning report:', REPORT_PATH)
print('Final shape:', df_clean.shape)

Saved cleaned dataset: /Users/yuvrajchandravansi/Desktop/card-fraud-triage/data/processed/creditcard_clean.csv
Saved cleaning report: /Users/yuvrajchandravansi/Desktop/card-fraud-triage/artifacts/clean_report.json
Final shape: (283726, 31)


## 2.11 Cleaning conclusion

- Raw rows were retained unless a missing/infinite value or exact duplicate copy required removal.
- `Time` was kept and sorted ascending for chronological modeling.
- No scaling, encoding, log transform, or feature engineering was performed.
- `Class` was retained as the evaluation label only.
- The cleaned file and audit report are ready for EDA.

The GMM/scaler/model split remains unchanged: it is handled later, with the scaler and GMM fit only on legitimate training rows.